In [ ]:
# ══════════════════════════════════════════════════════════════════
# STAGE 9: DEPLOYMENT — STREAMLIT APP
# Project #11 — Loan Default Risk Classification
# ══════════════════════════════════════════════════════════════════
# Run: streamlit run stage9_app.py
# Requires: lgbm_model.pkl, rf_model.pkl, lr_model.pkl,
#           scaler.pkl, top_features.pkl, threshold.json
# ══════════════════════════════════════════════════════════════════

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import json
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score,
    precision_recall_curve,
    accuracy_score, precision_score, recall_score, f1_score
)

# ─── Page Config ──────────────────────────────────────────────────
st.set_page_config(
    page_title="Loan Default Risk Classifier",
    page_icon="🏦",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─── Custom CSS ───────────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&display=swap');
html, body, [class*="css"] { font-family: 'DM Sans', sans-serif; }

[data-testid="stSidebar"] { background: #0f1117; border-right: 1px solid #1e2130; }
[data-testid="stSidebar"] * { color: #e0e6f0 !important; }

div[data-testid="metric-container"] {
    background: #f8faff; border: 1px solid #dde3f0;
    border-radius: 10px; padding: 14px 18px;
    box-shadow: 0 2px 8px rgba(0,60,180,0.06);
}
.risk-high {
    background: #fff0f0; border: 2px solid #e53e3e; border-radius: 12px;
    padding: 20px 28px; text-align: center; color: #c53030;
    font-weight: 700; font-size: 1.5rem;
}
.risk-low {
    background: #f0fff4; border: 2px solid #38a169; border-radius: 12px;
    padding: 20px 28px; text-align: center; color: #276749;
    font-weight: 700; font-size: 1.5rem;
}
.info-box {
    background: #eef3ff; border-left: 4px solid #3b6fdb;
    border-radius: 0 8px 8px 0; padding: 10px 16px;
    font-size: 0.9rem; color: #1a2a5e; margin: 8px 0 16px 0;
}
</style>
""", unsafe_allow_html=True)


# ─── Load Models ──────────────────────────────────────────────────
@st.cache_resource
def load_models():
    needed = {
        "lgbm":      "lgbm_model.pkl",
        "rf":        "rf_model.pkl",
        "lr":        "lr_model.pkl",
        "scaler":    "scaler.pkl",
        "top_feats": "top_features.pkl",
        "threshold": "threshold.json",
    }
    artefacts, missing = {}, []
    for key, fname in needed.items():
        if os.path.exists(fname):
            if fname.endswith(".json"):
                with open(fname) as f:
                    artefacts[key] = json.load(f)
            else:
                artefacts[key] = joblib.load(fname)
        else:
            missing.append(fname)
    return artefacts, missing


artefacts, missing_files = load_models()
models_loaded = len(missing_files) == 0


# ─── Helpers ──────────────────────────────────────────────────────
def build_input(raw: dict, top_feats: list) -> pd.DataFrame:
    row = {f: 0.0 for f in top_feats}
    for k, v in raw.items():
        if k in row:
            row[k] = v
    return pd.DataFrame([row])[top_feats]


def gauge(prob: float):
    fig, ax = plt.subplots(figsize=(4, 2.2))
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.2, 1.1); ax.axis("off")
    theta_bg = np.linspace(np.pi, 0, 200)
    ax.plot(np.cos(theta_bg), np.sin(theta_bg), color="#e0e6f0", lw=18, solid_capstyle="round")
    end_a = np.pi - prob * np.pi
    theta_fg = np.linspace(np.pi, end_a, 200)
    col = "#e53e3e" if prob > 0.5 else "#d97706" if prob > 0.3 else "#38a169"
    ax.plot(np.cos(theta_fg), np.sin(theta_fg), color=col, lw=18, solid_capstyle="round")
    ax.annotate("", xy=(np.cos(end_a) * 0.75, np.sin(end_a) * 0.75), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color="#1a2240", lw=2.5))
    ax.plot(0, 0, "o", color="#1a2240", ms=7, zorder=5)
    ax.text(0, -0.15, f"{prob:.1%}", ha="center", fontsize=18, fontweight="bold", color=col)
    ax.text(0,  0.55, "Default\nProbability", ha="center", fontsize=8, color="#8b9bbf")
    fig.patch.set_alpha(0)
    return fig


# ─── Sidebar ──────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("## 🏦 Loan Default\nRisk Classifier")
    st.markdown("---")
    page = st.radio("Navigate", [
        "🔍 Predict Risk",
        "📊 Model Evaluation",
        "🧠 Explainability",
        "ℹ️ About",
    ], label_visibility="collapsed")

    st.markdown("---")
    if models_loaded:
        threshold = artefacts["threshold"]["threshold"]
        for m in ["LightGBM ✓", "Random Forest ✓", "Logistic Regression ✓"]:
            st.markdown(f"<small style='color:#6ee7b7'>● {m}</small>", unsafe_allow_html=True)
        st.markdown(f"<small style='color:#8b9bbf'>Decision threshold: **{threshold:.4f}**</small>",
                    unsafe_allow_html=True)
    else:
        st.error("Model files missing")
        for f in missing_files:
            st.markdown(f"<small style='color:#fc8181'>✗ {f}</small>", unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════════
# PAGE 1 — PREDICT RISK
# ══════════════════════════════════════════════════════════════════
if page == "🔍 Predict Risk":
    st.markdown("# 🔍 Loan Default Risk Prediction")
    st.markdown('<div class="info-box">Fill in the applicant profile. '
                'All three models predict; <strong>LightGBM</strong> (best recall) is the primary decision.</div>',
                unsafe_allow_html=True)

    if not models_loaded:
        st.error("Models not loaded. Place all .pkl / .json files in the same folder as this script.")
        st.stop()

    top_feats = artefacts["top_feats"]
    threshold = artefacts["threshold"]["threshold"]

    with st.form("applicant_form"):
        c1, c2, c3 = st.columns(3)

        with c1:
            st.markdown("**💰 Income & Credit**")
            AMT_INCOME_TOTAL = st.number_input("Annual Income", 10_000, 10_000_000, 200_000, step=5_000)
            AMT_CREDIT       = st.number_input("Loan Amount",   10_000, 10_000_000, 500_000, step=10_000)
            AMT_ANNUITY      = st.number_input("Monthly Annuity", 1_000,   500_000,  25_000, step=1_000)
            AMT_GOODS_PRICE  = st.number_input("Goods Price",        0,  10_000_000, 450_000, step=10_000)

        with c2:
            st.markdown("**👤 Personal**")
            AGE_YEARS       = st.slider("Age (years)", 18, 70, 35)
            YEARS_EMPLOYED  = st.slider("Years Employed", 0, 40, 5)
            CNT_FAM_MEMBERS = st.slider("Family Members", 1, 10, 3)
            EXT_SOURCE_2    = st.slider("External Score 2 (0–1)", 0.0, 1.0, 0.55, 0.01)
            EXT_SOURCE_3    = st.slider("External Score 3 (0–1)", 0.0, 1.0, 0.50, 0.01)

        with c3:
            st.markdown("**📋 Loan Details**")
            REGION_RATING = st.selectbox("Region Risk Rating", [1, 2, 3])
            FLAG_OWN_CAR    = st.selectbox("Owns a Car?", ["Y", "N"])
            FLAG_OWN_REALTY = st.selectbox("Owns Realty?", ["Y", "N"])

        submitted = st.form_submit_button("⚡ Predict Default Risk", use_container_width=True)

    if submitted:
        # Derived features
        CREDIT_INCOME_RATIO   = AMT_CREDIT       / (AMT_INCOME_TOTAL + 1)
        ANNUITY_INCOME_RATIO  = AMT_ANNUITY      / (AMT_INCOME_TOTAL + 1)
        GOODS_CREDIT_RATIO    = AMT_GOODS_PRICE  / (AMT_CREDIT + 1)
        CREDIT_TERM           = AMT_CREDIT       / (AMT_ANNUITY + 1)
        EMPLOYED_TO_AGE_RATIO = YEARS_EMPLOYED   / (AGE_YEARS + 1)
        INCOME_PER_PERSON     = AMT_INCOME_TOTAL / (CNT_FAM_MEMBERS + 1)

        raw = {
            "AMT_INCOME_TOTAL":      AMT_INCOME_TOTAL,
            "AMT_CREDIT":            AMT_CREDIT,
            "AMT_ANNUITY":           AMT_ANNUITY,
            "AMT_GOODS_PRICE":       AMT_GOODS_PRICE,
            "DAYS_BIRTH":            -int(AGE_YEARS * 365),
            "DAYS_EMPLOYED":         -int(YEARS_EMPLOYED * 365),
            "AGE_YEARS":             AGE_YEARS,
            "YEARS_EMPLOYED":        YEARS_EMPLOYED,
            "CNT_FAM_MEMBERS":       CNT_FAM_MEMBERS,
            "EXT_SOURCE_2":          EXT_SOURCE_2,
            "EXT_SOURCE_3":          EXT_SOURCE_3,
            "CREDIT_INCOME_RATIO":   CREDIT_INCOME_RATIO,
            "ANNUITY_INCOME_RATIO":  ANNUITY_INCOME_RATIO,
            "GOODS_CREDIT_RATIO":    GOODS_CREDIT_RATIO,
            "CREDIT_TERM":           CREDIT_TERM,
            "EMPLOYED_TO_AGE_RATIO": EMPLOYED_TO_AGE_RATIO,
            "INCOME_PER_PERSON":     INCOME_PER_PERSON,
            "REGION_RATING_CLIENT":  REGION_RATING,
            "REGION_RATING_CLIENT_W_CITY": REGION_RATING,
        }

        X_in    = build_input(raw, top_feats)
        X_in_sc = artefacts["scaler"].transform(X_in)

        prob_lgbm = artefacts["lgbm"].predict_proba(X_in)[0][1]
        prob_rf   = artefacts["rf"].predict_proba(X_in)[0][1]
        prob_lr   = artefacts["lr"].predict_proba(X_in_sc)[0][1]

        pred_lgbm = int(prob_lgbm >= threshold)
        pred_rf   = int(prob_rf   >= 0.5)
        pred_lr   = int(prob_lr   >= 0.5)

        st.markdown("---")
        st.markdown("### Prediction Results")

        col_v, col_g = st.columns([1.3, 1])
        with col_v:
            if pred_lgbm == 1:
                st.markdown(
                    f'<div class="risk-high">⚠️ HIGH RISK — LIKELY DEFAULT<br>'
                    f'<span style="font-size:0.95rem;font-weight:400">'
                    f'Probability {prob_lgbm:.1%} ≥ threshold {threshold:.2f}</span></div>',
                    unsafe_allow_html=True)
            else:
                st.markdown(
                    f'<div class="risk-low">✅ LOW RISK — LIKELY SAFE<br>'
                    f'<span style="font-size:0.95rem;font-weight:400">'
                    f'Probability {prob_lgbm:.1%} < threshold {threshold:.2f}</span></div>',
                    unsafe_allow_html=True)
        with col_g:
            st.pyplot(gauge(prob_lgbm), use_container_width=True)

        st.markdown("#### All-Model Probabilities")
        ca, cb, cc = st.columns(3)
        ca.metric("LightGBM (Primary)", f"{prob_lgbm:.1%}", "⚠️ DEFAULT" if pred_lgbm else "✅ SAFE")
        cb.metric("Random Forest",      f"{prob_rf:.1%}",   "⚠️ DEFAULT" if pred_rf   else "✅ SAFE")
        cc.metric("Logistic Regression",f"{prob_lr:.1%}",   "⚠️ DEFAULT" if pred_lr   else "✅ SAFE")

        with st.expander("📐 Engineered Features"):
            st.table(pd.DataFrame({
                "Feature":  ["Credit-Income Ratio", "Annuity-Income Ratio", "Goods-Credit Ratio",
                             "Credit Term", "Employed-Age Ratio", "Income per Person"],
                "Value":    [f"{CREDIT_INCOME_RATIO:.3f}", f"{ANNUITY_INCOME_RATIO:.3f}",
                             f"{GOODS_CREDIT_RATIO:.3f}",  f"{CREDIT_TERM:.1f}",
                             f"{EMPLOYED_TO_AGE_RATIO:.3f}", f"{INCOME_PER_PERSON:,.0f}"],
            }).set_index("Feature"))

        st.markdown("#### 🧠 Local SHAP Explanation (LightGBM)")
        with st.spinner("Computing SHAP..."):
            try:
                exp_shap = shap.TreeExplainer(artefacts["lgbm"])
                sv_in    = exp_shap.shap_values(X_in)
                sv_in    = sv_in[1] if isinstance(sv_in, list) else sv_in
                base_val = (exp_shap.expected_value[1]
                            if isinstance(exp_shap.expected_value, list)
                            else exp_shap.expected_value)
                explanation = shap.Explanation(
                    values        = sv_in[0],
                    base_values   = base_val,
                    data          = X_in.iloc[0].values,
                    feature_names = top_feats,
                )
                shap.waterfall_plot(explanation, max_display=12, show=False)
                plt.title("SHAP Waterfall — This Applicant", fontweight="bold")
                plt.tight_layout()
                st.pyplot(plt.gcf(), use_container_width=True)
                plt.close("all")
            except Exception as e:
                st.info(f"SHAP unavailable: {e}")


# ══════════════════════════════════════════════════════════════════
# PAGE 2 — MODEL EVALUATION
# ══════════════════════════════════════════════════════════════════
elif page == "📊 Model Evaluation":
    st.markdown("# 📊 Model Evaluation & Comparison")
    st.markdown('<div class="info-box">Upload your evaluation CSV '
                '(columns: <code>y_true, y_proba_lgbm, y_proba_rf, y_proba_lr</code>).</div>',
                unsafe_allow_html=True)

    if not models_loaded:
        st.error("Models not loaded.")
        st.stop()

    uploaded = st.file_uploader("Upload eval_results.csv", type=["csv"])

    if uploaded:
        df_e = pd.read_csv(uploaded)
        required = {"y_true", "y_proba_lgbm", "y_proba_rf", "y_proba_lr"}
        if not required.issubset(df_e.columns):
            st.error(f"CSV must contain: {required}")
        else:
            thr = artefacts["threshold"]["threshold"]
            yt  = df_e["y_true"]
            proba = {"LightGBM": df_e["y_proba_lgbm"],
                     "Random Forest": df_e["y_proba_rf"],
                     "Logistic Regression": df_e["y_proba_lr"]}
            preds = {"LightGBM": (df_e["y_proba_lgbm"] >= thr).astype(int),
                     "Random Forest": (df_e["y_proba_rf"] >= 0.5).astype(int),
                     "Logistic Regression": (df_e["y_proba_lr"] >= 0.5).astype(int)}

            # Metrics table
            rows = []
            for nm in proba:
                rows.append({"Model": nm,
                    "Accuracy":  f"{accuracy_score(yt, preds[nm]):.4f}",
                    "Precision": f"{precision_score(yt, preds[nm], zero_division=0):.4f}",
                    "Recall":    f"{recall_score(yt, preds[nm]):.4f}",
                    "F1":        f"{f1_score(yt, preds[nm]):.4f}",
                    "ROC-AUC":   f"{roc_auc_score(yt, proba[nm]):.4f}"})
            st.markdown("### Summary Metrics")
            st.dataframe(pd.DataFrame(rows).set_index("Model"), use_container_width=True)

            # Confusion matrices
            st.markdown("### Confusion Matrices")
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            for ax, nm, cmap in zip(axes, preds, ["Blues", "Oranges", "Greens"]):
                cm = confusion_matrix(yt, preds[nm])
                ConfusionMatrixDisplay(cm, display_labels=["No Default", "Default"]).plot(
                    ax=ax, colorbar=False, cmap=cmap)
                ax.set_title(nm, fontweight="bold")
            plt.tight_layout()
            st.pyplot(fig, use_container_width=True); plt.close("all")

            # ROC curves
            st.markdown("### ROC-AUC Curves")
            fig, ax = plt.subplots(figsize=(8, 5))
            for nm, col in zip(proba, ["#1a56db", "#d97706", "#16a34a"]):
                fpr, tpr, _ = roc_curve(yt, proba[nm])
                auc = roc_auc_score(yt, proba[nm])
                ax.plot(fpr, tpr, label=f"{nm} (AUC={auc:.4f})", color=col, lw=2)
            ax.plot([0,1],[0,1],"k--",lw=1); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
            ax.set_title("ROC Curves", fontweight="bold"); ax.legend(); ax.grid(alpha=0.25)
            plt.tight_layout(); st.pyplot(fig, use_container_width=True); plt.close("all")

            # PR curves
            st.markdown("### Precision-Recall Curves")
            fig, ax = plt.subplots(figsize=(8, 5))
            for nm, col in zip(proba, ["#1a56db", "#d97706", "#16a34a"]):
                prec, rec, _ = precision_recall_curve(yt, proba[nm])
                ax.plot(rec, prec, label=nm, color=col, lw=2)
            ax.axhline(yt.mean(), color="red", ls="--", label=f"Baseline ({yt.mean():.2f})")
            ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
            ax.set_title("Precision-Recall Curves", fontweight="bold"); ax.legend(); ax.grid(alpha=0.25)
            plt.tight_layout(); st.pyplot(fig, use_container_width=True); plt.close("all")

    else:
        st.info("Generate the CSV from your notebook:\n```python\n"
                "pd.DataFrame({'y_true': y_test.values, 'y_proba_lgbm': y_proba_lgbm,\n"
                "              'y_proba_rf': y_proba_rf, 'y_proba_lr': y_proba_lr})"
                ".to_csv('eval_results.csv', index=False)\n```")


# ══════════════════════════════════════════════════════════════════
# PAGE 3 — EXPLAINABILITY
# ══════════════════════════════════════════════════════════════════
elif page == "🧠 Explainability":
    st.markdown("# 🧠 Model Interpretation & Explainability")
    st.markdown('<div class="info-box">Upload <code>X_test.csv</code> (top 50 feature columns) '
                'to render live SHAP plots.</div>', unsafe_allow_html=True)

    if not models_loaded:
        st.error("Models not loaded."); st.stop()

    top_feats = artefacts["top_feats"]
    uploaded_X = st.file_uploader("Upload X_test.csv", type=["csv"])

    if uploaded_X:
        X_t = pd.read_csv(uploaded_X)
        for c in top_feats:
            if c not in X_t.columns:
                X_t[c] = 0.0
        X_t = X_t[top_feats]

        n    = st.slider("SHAP sample size", 100, min(2000, len(X_t)), 500, 100)
        mdl  = st.selectbox("Model", ["LightGBM", "Random Forest"])
        obj  = artefacts["lgbm"] if mdl == "LightGBM" else artefacts["rf"]

        if st.button("⚡ Compute SHAP Values", use_container_width=True):
            with st.spinner("Computing..."):
                try:
                    Xs   = X_t.sample(n=n, random_state=42)
                    expl = shap.TreeExplainer(obj)
                    svs  = expl.shap_values(Xs)
                    sv   = svs[1] if isinstance(svs, list) else svs

                    st.markdown("### Global Feature Importance (Mean |SHAP|)")
                    fig, ax = plt.subplots(figsize=(9, 6))
                    mi = np.abs(sv).mean(axis=0)
                    pd.Series(mi, index=top_feats).sort_values().tail(20).plot(
                        kind="barh", ax=ax, color="#1a56db", edgecolor="white")
                    ax.set_title(f"{mdl} — Top 20 by Mean |SHAP|", fontweight="bold")
                    ax.set_xlabel("Mean |SHAP Value|"); plt.tight_layout()
                    st.pyplot(fig, use_container_width=True); plt.close("all")

                    st.markdown("### SHAP Beeswarm Plot")
                    fig2 = plt.figure(figsize=(9, 6))
                    shap.summary_plot(sv, Xs, max_display=20, show=False)
                    plt.title(f"{mdl} — SHAP Beeswarm", fontweight="bold")
                    plt.tight_layout(); st.pyplot(fig2, use_container_width=True); plt.close("all")

                    st.markdown("### Dependence Plots (Top 3 Features)")
                    top3 = X_t.columns[np.argsort(mi)[::-1][:3]].tolist()
                    fig3, axes = plt.subplots(1, 3, figsize=(16, 5))
                    for ax, ft in zip(axes, top3):
                        shap.dependence_plot(ft, sv, Xs, ax=ax, show=False)
                        ax.set_title(ft, fontsize=10)
                    plt.suptitle("SHAP Dependence Plots", fontweight="bold")
                    plt.tight_layout(); st.pyplot(fig3, use_container_width=True); plt.close("all")
                    st.success("✅ Done!")
                except Exception as e:
                    st.error(f"SHAP error: {e}")
    else:
        st.info("Generate X_test.csv from your notebook:\n```python\nX_test.to_csv('X_test.csv', index=False)\n```")


# ══════════════════════════════════════════════════════════════════
# PAGE 4 — ABOUT
# ══════════════════════════════════════════════════════════════════
elif page == "ℹ️ About":
    st.markdown("# ℹ️ Project #11 — Loan Default Risk Classification")
    st.markdown("""
### Pipeline

| Stage | Description |
|-------|-------------|
| 1–4   | Data loading, EDA, cleaning, preprocessing |
| 5     | Feature engineering (8 financial ratios) + top 50 by mutual information |
| 6     | Model training: Logistic Regression, Random Forest, LightGBM |
| 7     | Confusion matrices, ROC-AUC, PR curves, CV, threshold tuning |
| 8     | SHAP: global summary, dependence plots, force/waterfall plots |
| **9** | **This Streamlit app** |

### Required Model Files
```
lgbm_model.pkl      rf_model.pkl      lr_model.pkl
scaler.pkl          top_features.pkl  threshold.json
```

### Export from notebook (end of Stage 8)
```python
import joblib, json
joblib.dump(lgbm,      "lgbm_model.pkl")
joblib.dump(rf,        "rf_model.pkl")
joblib.dump(lr,        "lr_model.pkl")
joblib.dump(scaler,    "scaler.pkl")
joblib.dump(top_feats, "top_features.pkl")
with open("threshold.json","w") as f:
    json.dump({"threshold": float(THRESHOLD)}, f)
```

### Run & Deploy
```bash
# Local
pip install -r requirements.txt
streamlit run stage9_app.py

# Streamlit Cloud (free)
# Push all files to GitHub → share.streamlit.io → New App
```

### Key Design Choices
- **Recall ≥ 0.70** optimised — a missed defaulter costs more than a false alarm
- **SMOTE** applied on training set only (no data leakage)
- **LightGBM** primary model — highest ROC-AUC and recall at tuned threshold
- **SHAP TreeExplainer** — exact, fast, regulatory-compliant explanations
    """)